# Introduction to LangChain

## Initial setup

### Set API key for Groq
Click [here](https://console.groq.com/keys) to create API key for Groq, if not already created.

In [19]:
import os, json, re, getpass
from dotenv import load_dotenv

load_dotenv( override=True)

False

In [20]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [21]:
# if "TEST_API_KEY" not in os.environ:
#     os.environ["TEST_API_KEY"] = getpass.getpass("TEST API Key: ")

In [22]:
if os.environ["GROQ_API_KEY"]:
    print(f"Groq API Key exists and begins {os.environ["GROQ_API_KEY"][:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


## LangChain Components

### LLM / ChatModel

**Note** on **init_chat_model**: init_chat_model is just a helper method and under the hood, it will still be calling the specific Chat models only (like ChatOpenAI etc.). The only benefit of using init_chat_model is that the initialization is standard across providers, which is useful. 

See the source code of this method here for better details. Note that line 79 has init_chat_model() function and if model and model_provider are specified (which we do in class), then it returns an instance of _ConfigurableModel class (line 332). Then this class definition (line 554), returns the model in line 610 using _init_chat_model_helper() method. This method definition (line 339) actually returns ChatOllama() model instance (line 400). So it's the same thing.


In [23]:
#Using LangChain
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq")

In [24]:
llm_response = llm.invoke("what is ECG?")
llm_response

AIMessage(content='**Electrocardiogram (ECG or EKG)**  \n\n| Aspect | What it means |\n|--------|----------------|\n| **Full name** | **Electro‑cardio‑gram** – a recording of the electrical activity of the heart. |\n| **Purpose** | Detects how fast and rhythmically the heart beats, and reveals problems with the heart muscle, its conduction system, blood supply, electrolyte balance, or drug effects. |\n| **How it works** | The heart’s muscle cells generate tiny electric currents each time they contract. By placing electrodes on the skin, those currents are captured, amplified, and plotted as voltage (vertical axis) versus time (horizontal axis). |\n| **Typical set‑up** | 12‑lead ECG is the standard clinical format: <br>• **Limb leads** (I, II, III, aVR, aVL, aVF) – electrodes on both arms and both legs. <br>• **Pre‑cordial (chest) leads** (V1‑V6) – electrodes placed at specific points on the chest. <br>These 12 views together give a 3‑dimensional picture of the heart’s electrical activi

In [25]:
print("type of response", type(llm_response))

type of response <class 'langchain_core.messages.ai.AIMessage'>


In [26]:
# print(llm_response.content)
# display(llm_response.response_metadata)
display(llm_response.usage_metadata)

{'input_tokens': 75, 'output_tokens': 1286, 'total_tokens': 1361}

In [27]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
                  Ensure proper sentence structure, clarity, and readability.\
                  Retain the core message of the original text while making the necessary corrections"),
    HumanMessage(content="hey can you send me that report by tomorrow thx"),
]

ai_response = llm.invoke(messages)
print(ai_response.content)

Hey, can you send me that report by tomorrow? Thanks.


In [28]:
ai_response

AIMessage(content='Hey, can you send me that report by tomorrow? Thanks.', additional_kwargs={'reasoning_content': 'We need to correct the text: "hey can you send me that report by tomorrow thx". Need to correct spelling, grammar, punctuation, sentence structure, clarity, readability. Retain core message.\n\nCorrect version: "Hey, can you send me that report by tomorrow? Thanks."\n\nProbably add proper capitalization, punctuation.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 127, 'total_tokens': 215, 'completion_time': 0.182882007, 'prompt_time': 0.005854551, 'queue_time': 0.413095417, 'total_time': 0.188736558, 'completion_tokens_details': {'reasoning_tokens': 66}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--88f835b2-959e-429a-b498-094f7cb7a55c-0', usage_metadata={'input_tokens': 127, 'output_tokens': 88, 'total_tokens': 215})

In [29]:
# followup conversation
messages.append(ai_response)

In [30]:
#Ask a follow-up question
messages.append(HumanMessage(content="can you make the tone a bit informal"))

In [31]:
ai_response = llm.invoke(messages)
# print(ai_response)
print(ai_response.content)

Hey, could you shoot me that report by tomorrow? Thanks!


#### Doing without LangChain

In [32]:
#Without LangChain - how would we initialize our LLM?
from openai import OpenAI

model_name = "openai/gpt-oss-120b"
llm_api = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

In [33]:
# #Below is LangChain's messages format
# messages = [

#     SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
#                   Ensure proper sentence structure, clarity, and readability.\
#                   Retain the core message of the original text while making the necessary corrections"),
#     HumanMessage(content="hey can you send me that report by tomorrow thx"),
# ]

#Below is messages in OpenAI format
messages_openai = [
    {'role':"system", 'content':"Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
     Ensure proper sentence structure, clarity, and readability.\
     Retain the core message of the original text while making the necessary corrections"},
     
     {'role':"user", 'content':"hey can you send me that report by tomorrow thx"}
]


In [34]:
ai_response_openai = llm_api.chat.completions.create(model= model_name,
                                messages=messages_openai)

In [35]:
ai_response_openai

ChatCompletion(id='chatcmpl-4c55a34c-ed6f-4546-b8d5-78430d049737', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hey, can you send me that report by tomorrow? Thanks.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks: "hey can you send me that report by tomorrow thx". They want correction. We need to detect and correct spelling, grammar, punctuation, improve sentence structure, clarity, readability, retain core message. So corrected: "Hey, can you send me that report by tomorrow? Thanks."\n\nReturn corrected text.'))], created=1788843672, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_f73454f048', usage=CompletionUsage(completion_tokens=89, prompt_tokens=127, total_tokens=216, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=67, rej

In [36]:
ai_response_openai_formatted = ai_response_openai.choices[0].message.content
print(ai_response_openai_formatted)

Hey, can you send me that report by tomorrow? Thanks.


In [37]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'}]

In [38]:
#Append the AI message
messages_openai.append(
    {'role': "assistant",
     'content': ai_response_openai_formatted}
)

In [39]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Hey, can you send me that report by tomorrow? Thanks.'}]

In [40]:
#Ask a follow-up question
#LangChain version below
# messages.append(HumanMessage(content="can you make the tone a bit informal"))

#OpenAI version below
messages_openai.append(
    {'role':"user",
     'content':"can you make the tone a bit informal"}
)

In [41]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Hey, can you send me that report by tomorrow? Thanks.'},
 {'role': 'user', 'content': 'can you make the tone a bit informal'}]

In [42]:
ai_response_openai = llm_api.chat.completions.create(
    model=model_name,
    messages=messages_openai
)

In [43]:
type(ai_response_openai)

openai.types.chat.chat_completion.ChatCompletion

In [44]:
print(ai_response_openai.choices[0].message.content)

Hey, could you send me that report by tomorrow? Thanks!


In [45]:
#Explain concept of context length here - Context length = input tokens + completion tokens 

### Output Parsers

In [46]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

parser = StrOutputParser()

*StrOutputParser* is a runnable object.

In [47]:
result = llm.invoke(messages)

parser.invoke(result)

'Hey, could you shoot me that report by tomorrow? Thanks!'

In [48]:
result.content ## NOT DOING THIS ANYMORE, USING THE PARSER INSTEAD

'Hey, could you shoot me that report by tomorrow? Thanks!'

In [49]:
result

AIMessage(content='Hey, could you shoot me that report by tomorrow? Thanks!', additional_kwargs={'reasoning_content': 'We need to respond with revised text, informal tone, corrected. Original: "Hey, can you send me that report by tomorrow? Thanks." Need informal tone: maybe "Hey, could you shoot me that report by tomorrow? Thanks!" Keep corrections. Provide revised.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 158, 'total_tokens': 235, 'completion_time': 0.158815429, 'prompt_time': 0.021585653, 'queue_time': 0.322389124, 'total_time': 0.180401082, 'completion_tokens_details': {'reasoning_tokens': 55}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--8347b7df-0fb5-4976-8a10-0a7787a73c8d-0', usage_metadata={'input_tokens': 158, 'output_tokens': 77, 'total_tokens': 235})

In [50]:
messages = [
    SystemMessage(content="""You are an expert in writing analysis. You will receive a message from a user, and your job is to evaluate the text based on the following attributes:
1. clarity: Is the message clear, or unclear?
2. grammar_quality: Are there any grammatical issues? Possible values: correct, minor issues, major issues.
3. tone: Analyze whether the tone is neutral, formal, or informal.
4. suggestions: Offer brief improvement suggestions for clarity, grammar, or tone.

Return a structured JSON object with these four attributes. Wrap the JSON between ```json tags"""),
    HumanMessage(content="Hey, could you please send me that report by tomorrow? Thank you.")
]
response = llm.invoke(messages)

In [51]:
response

AIMessage(content='```json\n{\n  "clarity": "clear",\n  "grammar_quality": "correct",\n  "tone": "informal",\n  "suggestions": "If a more formal tone is desired, replace \\"Hey\\" with \\"Hello\\" or \\"Dear [Name]\\" and consider adding a closing such as \\"Best regards\\"."\n}\n```', additional_kwargs={'reasoning_content': 'The user asks to evaluate the text based on attributes. Need to produce JSON with clarity, grammar_quality, tone, suggestions. Evaluate the given sentence: "Hey, could you please send me that report by tomorrow? Thank you."\n\nClarity: clear. Grammar: correct. Tone: informal (Hey, thank you). Suggestions: maybe make more formal if needed. Provide JSON.'}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 203, 'total_tokens': 360, 'completion_time': 0.327675062, 'prompt_time': 0.075149982, 'queue_time': 0.538969545, 'total_time': 0.402825044, 'completion_tokens_details': {'reasoning_tokens': 77}}, 'model_name': 'openai/gpt-oss-120b', 'sy

In [52]:
print(response.content)

```json
{
  "clarity": "clear",
  "grammar_quality": "correct",
  "tone": "informal",
  "suggestions": "If a more formal tone is desired, replace \"Hey\" with \"Hello\" or \"Dear [Name]\" and consider adding a closing such as \"Best regards\"."
}
```


In [53]:
json_response = JsonOutputParser().invoke(response.content)
json_response

{'clarity': 'clear',
 'grammar_quality': 'correct',
 'tone': 'informal',
 'suggestions': 'If a more formal tone is desired, replace "Hey" with "Hello" or "Dear [Name]" and consider adding a closing such as "Best regards".'}

In [54]:
var1 = '{"clarity": "unclear"}'
print(var1)

{"clarity": "unclear"}


In [55]:
var1

'{"clarity": "unclear"}'

In [56]:
# JsonOutputParser().invoke(var1)

In [57]:
type(json_response)

dict

In [58]:
type(var1)

str

In [59]:
print(response.content)

```json
{
  "clarity": "clear",
  "grammar_quality": "correct",
  "tone": "informal",
  "suggestions": "If a more formal tone is desired, replace \"Hey\" with \"Hello\" or \"Dear [Name]\" and consider adding a closing such as \"Best regards\"."
}
```


In [60]:
json_response['clarity']

'clear'

### Chain (LCEL)

In [61]:
chain = llm | JsonOutputParser()

chain.invoke(messages)

{'clarity': 'clear',
 'grammar_quality': 'correct',
 'tone': 'informal',
 'suggestions': 'Consider a slightly more formal phrasing, e.g., "Please send the report by tomorrow. Thank you."'}

### PromptTemplate

In [62]:
# my_str = "Hello world, this is the tone - {tone}"

In [63]:
# my_str.format(tone = "Happy")

In [64]:
# tone = "Happy"
# my_str = f"Hello world, this is the tone - {tone}"
# print(my_str)

In [65]:
my_str = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

In [66]:
print(my_str.format(tone="Happy", communication_style = "Formal"))

Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: Happy

Communication Style: Formal

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style.


In [67]:
from langchain_core.prompts import ChatPromptTemplate
system_message_template = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

#Defining a ChatPromptTemplate
template = ChatPromptTemplate([
    ("system", system_message_template),
    ("human", "{user_input}"),
])

In [68]:
template.input_variables

['communication_style', 'tone', 'user_input']

In [69]:
#How does this work without LCEL = LangChain Expression LangChain

In [70]:
tone = 'Rewrite the message in a professional, polite, and structured manner. \
Suitable for business emails, official reports, or any context requiring formality and respect.'

communication_style = 'Messages should be clear, structured, and formal or neutral depending on the context. \
Introductions, conclusions, and appropriate sign-offs should be added if missing.'

user_input = """Can u send me the data by eod pls?"""

In [71]:
#First step
formatted_template = template.invoke({"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
formatted_template

ChatPromptValue(messages=[SystemMessage(content='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.\n\nCommunication Style: Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can u send me the data by eod pls?', additional_kwargs={}, response_metadata={})])

In [72]:
type(formatted_template)

langchain_core.prompt_values.ChatPromptValue

In [73]:
#Second step
response = llm.invoke(formatted_template)

In [74]:
type(response)

langchain_core.messages.ai.AIMessage

In [75]:
response.content

'**Subject:** Request for Data by End of Day  \n\nDear [Recipient’s Name],\n\nI hope you are well.  \n\nCould you please send me the requested data by the end of the business day today? Your prompt assistance would be greatly appreciated.\n\nThank you for your cooperation.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [76]:
#Third step
final_parsed_result = StrOutputParser().invoke(response)
final_parsed_result

'**Subject:** Request for Data by End of Day  \n\nDear [Recipient’s Name],\n\nI hope you are well.  \n\nCould you please send me the requested data by the end of the business day today? Your prompt assistance would be greatly appreciated.\n\nThank you for your cooperation.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [77]:
# #With LCEL
# proof_read_chain = template | llm ##RunnableSequence
# final_response = proof_read_chain.invoke(
#     {"communication_style": communication_style,
#     "tone":tone,
#     "user_input":user_input
# })
# final_response
# type(final_response)

In [78]:
#With LCEL
proof_read_chain = template | llm | StrOutputParser()
proof_read_chain

ChatPromptTemplate(input_variables=['communication_style', 'tone', 'user_input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['communication_style', 'tone'], input_types={}, partial_variables={}, template='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: {tone}\n\nCommunication Style: {communication_style}\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, template='{user_input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x135b4c2d0>, async_client=<groq.

In [79]:
type(proof_read_chain)

langchain_core.runnables.base.RunnableSequence

In [80]:
final_response = proof_read_chain.invoke(
    {"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
final_response

'Subject: Request for Data Submission by End of Day\n\nDear [Recipient’s Name],\n\nI hope you are doing well.\n\nCould you please send me the requested data by the end of the business day today? Your prompt attention to this matter would be greatly appreciated.\n\nThank you for your assistance.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [81]:
# proof_read_chain.input_schema.model_json_schema()

In [82]:
tone_map = {
    "Formal": "Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.",
    "Informal": "Rewrite in a casual, friendly, and conversational style. Appropriate for personal communications, friendly chats, or informal emails.",
    "Neutral": "Rewrite in a balanced tone that is neither overly formal nor too casual. Suitable for most general communications where a middle-ground tone is required."
}
communication_style_map = {
    "Email": "Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.",
    "General": "This covers most forms of communication and will aim for clarity and coherence. The tone can vary as per the user's choice.",
    "Instant Messaging": "Focus on brevity, clarity, and informality, using conversational phrasing suitable for quick back-and-forth exchanges.",
    "Business Instant Messaging": "Maintain a professional but conversational tone. Messages should be concise and efficient, avoiding unnecessary formalities but keeping the language respectful."
}

In [83]:
tone = "Formal" # Formal, Informal, Neutral
communication_style = "Email" # Email, General, Instant Messaging, Business Instant Messaging
user_input = """Can u send me the data by eod pls?"""

chain_output = proof_read_chain.invoke(dict(
    tone=tone_map[tone],
    communication_style = communication_style_map[communication_style],
    user_input = user_input
))
print(chain_output)

**Revised Message**

---

Dear [Recipient’s Name],

Could you please send me the requested data by the end of the business day?  

Thank you for your assistance.

Best regards,  
[Your Name]  
[Your Position]  
[Your Company]  
[Contact Information]  
